## 5.1 Setup

In [ ]:
import os

REPO_URL = "https://github.com/meriem200512365/Chat-boot-cegedim.git"
REPO_DIR = "Chat-boot-cegedim"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}


In [ ]:
%pip install -q -r requirements.txt

In [ ]:
!python scripts/index_data.py

## 5.2 Clé API Groq (saisie sécurisée)

Deux options :
- **Colab Secrets** (recommandé) : icône clé 🔑 dans la barre latérale
  gauche -> ajouter un secret nommé `GROQ_API_KEY` -> activer l'accès au
  notebook.
- Sinon, saisie masquée via `getpass` (rien n'est affiché ni sauvegardé
  dans le notebook).

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("Cle recuperee depuis les secrets Colab.")
except Exception:
    from getpass import getpass
    os.environ["GROQ_API_KEY"] = getpass("Colle ta cle GROQ_API_KEY (saisie masquee) : ")


## 5.3 Import des deux modes du projet

In [ ]:
import sys
sys.path.insert(0, ".")

from src.chatbot.chatbot import repondre as repondre_direct
from src.generation.rag_chatbot import repondre_rag


## 5.4 Jeu de questions test

In [ ]:
QUESTIONS_TEST = [
    "je veux gerer les actes RO",
    "comment parametrer un devis web",
    "ou trouver la gestion des cheques",
    "je cherche a annuler un cheque, comment faire",
    "acces aux referentiels de prestations",
    "question totalement hors sujet, sans rapport",
]


## 5.5 Exécution côte à côte, avec mesure de latence

In [ ]:
import time

lignes = []

for question in QUESTIONS_TEST:
    # --- Mode direct ---
    t0 = time.time()
    rep_direct = repondre_direct(question)
    latence_direct = time.time() - t0

    # --- Mode RAG + LLM ---
    t0 = time.time()
    rep_rag = repondre_rag(question)
    latence_rag = time.time() - t0

    lignes.append({
        "question": question,
        "reponse_directe": rep_direct["message"],
        "latence_directe_s": round(latence_direct, 3),
        "reponse_rag": rep_rag["reponse"],
        "latence_rag_s": round(latence_rag, 3),
        "sources_rag": [s["path_str"] for s in rep_rag["chemins_sources"]],
    })


In [ ]:
import pandas as pd

df_comparaison = pd.DataFrame(lignes)
pd.set_option("display.max_colwidth", None)
df_comparaison[["question", "reponse_directe", "reponse_rag"]]


## 5.6 Comparaison des latences

In [ ]:
import matplotlib.pyplot as plt

x = range(len(df_comparaison))
plt.figure(figsize=(10, 4))
plt.bar([i - 0.2 for i in x], df_comparaison["latence_directe_s"], width=0.4, label="Mode direct")
plt.bar([i + 0.2 for i in x], df_comparaison["latence_rag_s"], width=0.4, label="Mode RAG + LLM")
plt.xticks(list(x), [q[:25] + "..." for q in df_comparaison["question"]], rotation=60, ha="right")
plt.ylabel("Latence (secondes)")
plt.title("Latence : reponse directe vs RAG + LLM")
plt.legend()
plt.tight_layout()
plt.show()

print(df_comparaison[["latence_directe_s", "latence_rag_s"]].describe())


## 5.7 Vérification anti-hallucination

Le prompt système de `rag_chatbot.py` interdit explicitement au LLM
d'inventer un chemin. On vérifie ici que chaque chemin cité dans la
réponse générée existe bien dans `chemins_sources` (donc dans le vrai
menu).

In [ ]:
def chemins_cites_sont_valides(reponse_dict):
    chemins_valides = {s["path_str"] for s in reponse_dict["chemins_sources"]}
    reponse_texte = reponse_dict["reponse"]
    # heuristique simple : verifie qu'au moins un chemin source apparait tel quel
    return any(chemin in reponse_texte for chemin in chemins_valides) if chemins_valides else None

for question in QUESTIONS_TEST:
    rep = repondre_rag(question)
    ok = chemins_cites_sont_valides(rep)
    print(f"{'OK' if ok else '??'}  {question}")
